### Fe構造の階層クラスタリング

**データ取得から可視化**

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pylab as plt
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 10)


クラスタリングは以下のように行えます。

In [ ]:
# データ取得（読み込み）
g_df = pd.read_csv("../data_calculated/Fe2_descriptor.csv")
g_df

In [ ]:
g_descriptor_names = ['a0.70_rp2.40', 'a0.70_rp3.00', 'a0.70_rp3.60', 'a0.70_rp4.20',
                     'a0.70_rp4.80', 'a0.70_rp5.40']
g_Xraw = g_df[g_descriptor_names].values

# データプリプロセス
g_scaler = StandardScaler()
g_scaler.fit(g_Xraw)
g_X = g_scaler.transform(g_Xraw)
g_labels = g_df["key"].values.tolist()


可視化を含めて解析していきます。
二次元で可視化するためにPCAにより次元圧縮を行っています。
これも次元圧縮の用途の一つです。

別のクラスタリング手法である
階層クラスタリングも行うことができます。

これは各サンプル点間の距離を計算し、最短の距離にある点を順につなげていくことで樹形図を作成する手法です。
距離の計算手法やグループ間の距離の計算手法がそれぞれ幾つか存在し、目的に応じて最適な手法を選択します。

In [ ]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import pearsonr
import numpy as np
from scipy.spatial.distance import squareform
import seaborn as sns

def make_dendrogram(X, labels, metric='euclidean',figsize=(10,10)):
    """階層クラスタリングを行う。

    Args:
        X (np.ndarray): normalized explanatory variables
        labels ([str]): 説明変数名
        metric (str, optional): metric名. Defaults to 'euclidean'.
        figsize (tuple, optional): 図のサイズ. Defaults to (10,10).

    Returns:
        nd.pdarray: pairdistance.
        dendrogram: denddrogram instance
    """
    print("X.shape",X.shape)
    print("labels", len(labels), labels)
    if True:
        if metric=="minus_abs_pearson":
            print(" metric", metric)
            df_tmp = pd.DataFrame(X)
            if True:
                corr = 1- np.abs(df_tmp.T.corr()) # DataFrame can calculate all the pearson's correlations
            else:
                corr = []
                for x1 in X:
                    corr1 = []
                    for x2 in X:
                        corr1.append(1.0-np.abs(pearsonr(x1,x2)[0]))
                    corr.append(corr1)
                corr = np.array(corr)
                for i in range(corr.shape[0]): # set the diagonal parts explicitly zero because of numerical precision.
                    corr[i,i] = 0.0
            print("corr",corr)
            print("corr.shape", corr.shape)
            pairdistance = squareform(corr)
            print("pairdistance", pairdistance.shape)
        else:
            print("metric",metric)
            pairdistance = pdist(X, metric=metric)  # calculate pair distance
        Z = linkage(pairdistance) # pairistance is 1D array
    else:
        Z = linkage(X,metric=metric) # 2D array
        
    fig, ax= plt.subplots(figsize=figsize)
    tree = dendrogram(Z, labels=labels, orientation="left", ax=ax)
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig("image_executed/Fe2_dendrogram.png")
    fig.show()
    return pairdistance, tree

def make_pairdistance_matrix(pairdistance, labels, tree, figsize=(10,10)):
    """pair distanceを示すDataFrameを作成する。

    Args:
        pairdistance (nd.ndarray): pair distance。
        labels ([str]]): 説明変数名のリスト。
        tree (dendrogram): dendrogram instance
        figsize (tuple, optional): 図のサイズ. Defaults to (10,10).
    """
    pairdistance_matrix = squareform(pairdistance)
    df_pdistmatrix = pd.DataFrame(pairdistance_matrix, index=labels, columns=labels)
    df_pdistmatrix
    
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(df_pdistmatrix.loc[tree["ivl"],tree["ivl"]], ax=ax)
    

In [ ]:
g_pairdistance, g_tree = make_dendrogram(g_X, g_labels,figsize=(5,10))
make_pairdistance_matrix(g_pairdistance, g_labels, g_tree)

In [ ]:
g_figsize=(5,5)
g_pairdistance, g_tree = make_dendrogram(g_X.T, g_descriptor_names, metric="minus_abs_pearson", figsize=g_figsize)
make_pairdistance_matrix(g_pairdistance, g_descriptor_names, g_tree, figsize=g_figsize)

In [ ]:
g_df.set_index("key", drop=True)

In [ ]:
import seaborn as sns
    
g_map = sns.clustermap(g_df.set_index("key")[g_descriptor_names], 
                       metric="euclidean", figsize=(10,10) ) # can't use ax=

# 補足

丸く表示するライブラリもあります。


```
pip install pycirclize 
```

In [ ]:
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr

import numpy as np
import pandas as pd


def make_dendrogram(
    X,
    labels,
    metric="euclidean",
):
    """
    pycirclize 用 hierarchical clustering

    Args:
        X (np.ndarray):
            normalized explanatory variables

        labels (list[str]):
            label names

        metric (str):
            distance metric

    Returns:
        pairdistance (np.ndarray):
            condensed distance matrix

        Z (np.ndarray):
            scipy linkage matrix
    """

    print("X.shape", X.shape)
    print("len(labels)", len(labels))

    # -----------------------------
    # distance matrix
    # -----------------------------
    if metric == "minus_abs_pearson":

        print("metric =", metric)

        df_tmp = pd.DataFrame(X)

        corr = 1 - np.abs(df_tmp.T.corr())
        corr = corr.to_numpy().copy()
        np.fill_diagonal(corr, 0.0)

        print("corr.shape", corr.shape)

        # square matrix -> condensed vector
        pairdistance = squareform(corr)

    else:

        print("metric =", metric)

        pairdistance = pdist(X, metric=metric)

    print("pairdistance.shape", pairdistance.shape)

    # -----------------------------
    # linkage
    # -----------------------------
    Z = linkage(pairdistance, method="ward")

    print("Z.shape", Z.shape)

    return pairdistance, Z

g_pairdistance, g_Z = make_dendrogram(
    g_X,
    g_labels,
    metric="minus_abs_pearson"
)

以下を実行するには
https://moshi4.github.io/pyCirclize/#installation
を参照せよ。


In [ ]:
from pathlib import Path
from pycirclize import Circos
from scipy.cluster.hierarchy import to_tree


def linkage_to_newick(Z, labels):
    tree = to_tree(Z, rd=False)

    def build_newick(node):
        if node.is_leaf():
            return str(labels[node.id])

        left = build_newick(node.left)
        right = build_newick(node.right)

        left_dist = node.dist - node.left.dist
        right_dist = node.dist - node.right.dist

        return f"({left}:{left_dist:.6f},{right}:{right_dist:.6f})"

    return build_newick(tree) + ";"


def make_circular_dendrogram(
    Z,
    labels,
    tree_file="image_executed/tmp_tree.nwk",
    r_lim=(20, 100),
    leaf_label_size=6,
):
    """
    pycirclize で円形 dendrogram を作る。
    plot はしない。
    """
    newick = linkage_to_newick(Z, labels)

    tree_file = Path(tree_file)
    tree_file.parent.mkdir(parents=True, exist_ok=True)
    tree_file.write_text(newick)

    circos, tv = Circos.initialize_from_tree(
        tree_file,
        r_lim=r_lim,
        leaf_label_size=leaf_label_size,
        line_kws=dict(lw=1.0),
    )

    return circos, tv
    

circos, tv = make_circular_dendrogram(
    g_Z,
    g_labels,
    r_lim=(10, 50),
    leaf_label_size=12
)
fig = circos.plotfig()
